In [ ]:
# Standard libraries
import os
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV,cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, mutual_info_regression, f_classif, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

# Other utilities
from scipy.stats import randint, uniform
import joblib

# Standard libraries
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Scikit-learn utilities
from sklearn.impute import SimpleImputer

In [ ]:
path = "../../data/processed/"
dfs_processed = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs_processed[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs_processed[name].shape}")

sites = dfs_processed[list(dfs_processed.keys())[6]]

regiones = sites['HERlvl1Code'].drop_duplicates().tolist()
len(regiones)


Loaded 03_CLEAN_COMPLETE_DF with shape (49863, 174)
Loaded 03_CLEAN_COMPLETE_DF_02 with shape (49231, 623)
Loaded 03_COMPLETE_TEST with shape (43568, 3)
Loaded 03_COMPLETE_TRAIN with shape (49441, 2332)
Loaded 03_COMPLETE_TRAIN_2 with shape (49231, 2849)
Loaded clean_train with shape (43568, 4)
Loaded dep_codes with shape (49231, 12)
Loaded dep_test with shape (5063, 12)
Loaded taxones_pressure with shape (5663, 2333)
Loaded taxones_pressure_epm_predict with shape (5663, 2850)
Loaded taxones_pressure_epm_train with shape (43568, 2853)
Loaded taxones_pressure_predict with shape (5663, 2333)
Loaded taxones_pressure_train with shape (43568, 2336)


22

In [ ]:

path = "../../notebooks/06_cb_regression/dfs_taxon_p/"
dfs = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs[name].shape}")

from types import SimpleNamespace

# Después de llenar `dfs`:
d = SimpleNamespace(**dfs)

Loaded df_1 with shape (1485, 109)
Loaded df_10 with shape (3926, 148)
Loaded df_11 with shape (1289, 163)
Loaded df_12 with shape (4677, 176)
Loaded df_13 with shape (1003, 172)
Loaded df_14 with shape (7742, 161)
Loaded df_15 with shape (1223, 160)
Loaded df_16 with shape (379, 113)
Loaded df_17 with shape (803, 151)
Loaded df_18 with shape (1164, 152)
Loaded df_19 with shape (500, 134)
Loaded df_2 with shape (352, 97)
Loaded df_20 with shape (508, 209)
Loaded df_21 with shape (2663, 184)
Loaded df_22 with shape (152, 175)
Loaded df_3 with shape (4996, 141)
Loaded df_4 with shape (596, 141)
Loaded df_5 with shape (2313, 131)
Loaded df_6 with shape (2134, 142)
Loaded df_7 with shape (524, 107)
Loaded df_8 with shape (548, 123)
Loaded df_9 with shape (10254, 169)


In [ ]:

def train_catboost_region(
    cleandf: pd.DataFrame,
    target: str = 'IBD',
    test_size: float = 0.20,
    random_state: int = 42,
    early_stopping_rounds: int = 200,
    cat_params: dict | None = None,
):
    """
    Entrena CatBoost para una región usando cleandf.
    - Separa train (con target) y score (sin target)
    - Preprocesa (imputación num/cat + OHE)
    - Entrena con early stopping
    - Regresa: modelo (Pipeline), métricas, scored_df (filas sin target con predicción)
    """
    # 1) separar train / score
    df_train = cleandf[cleandf[target].notna()].copy()
    df_score = cleandf[cleandf[target].isna()].copy()

    # X / y
    drop_cols = [c for c in ['IBD','IBD_EQR','IBD_EQR_Status'] if c in cleandf.columns]
    X = df_train.drop(columns=drop_cols, errors='ignore')
    y = df_train[target].astype(float)

    # split
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=random_state)

    # 2) columnas num/cat
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    # preprocesamiento
    pre = ColumnTransformer([
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
        ]), num_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='(missing)')),
            ('ohe', OneHotEncoder(handle_unknown='ignore')),
        ]), cat_cols)
    ])

    # 3) modelo CatBoost (parámetros por defecto + override opcional)
    base_params = dict(
        depth=6, learning_rate=0.05, n_estimators=3000,
        loss_function='RMSE', random_state=random_state, verbose=0
    )
    if cat_params:
        base_params.update(cat_params)

    cb = Pipeline([
        ('pre', pre),
        ('model', CatBoostRegressor(**base_params))
    ])

    # 4) ajustar: primero ajustamos el preprocesador para armar matrices, luego el modelo con early stopping
    Xtr_proc = pre.fit_transform(X_tr)
    Xte_proc = pre.transform(X_te)
    cb.named_steps['model'].fit(
        Xtr_proc, y_tr,
        eval_set=(Xte_proc, y_te),
        use_best_model=True,
        early_stopping_rounds=early_stopping_rounds
    )

    # 5) métricas en hold-out y train (usando el pipeline para que transforme igual)
    r2_tr  = cb.score(X_tr, y_tr)
    r2_te  = cb.score(X_te, y_te)
    pred_tr = cb.predict(X_tr); pred_te = cb.predict(X_te)
    metrics = {
        'R2_train': r2_tr,
        'R2_valid': r2_te,
        'MAE_train': mean_absolute_error(y_tr, pred_tr),
        'MAE_valid': mean_absolute_error(y_te, pred_te),
        'RMSE_train': mean_squared_error(y_tr, pred_tr),
        'RMSE_valid': mean_squared_error(y_te, pred_te),
        'best_iterations': int(cb.named_steps['model'].get_best_iteration() or base_params['n_estimators'])
    }

    # 6) predicciones para filas sin target de esta región (si existen)
    if not df_score.empty:
        X_score = df_score.drop(columns=drop_cols, errors='ignore')
        df_score[target + '_pred'] = cb.predict(X_score)
        scored_df = df_score
        
    else:
        scored_df = pd.DataFrame(columns=list(cleandf.columns) + [target + '_pred'])

    return cb, metrics, scored_df

In [ ]:
"""predicciones = []
all_metrics = {}  # guardará {region: métricas}

for region in regiones:
    attr = f"df_{int(region)}"           # df_1, df_2, df_18, ...
    if not hasattr(d, attr):
        # fallback por si tus nombres vienen con ceros: df_01
        attr = f"df_{str(region)}"
    if not hasattr(d, attr):
        raise AttributeError(f"No encontré {attr} en d")

    cleandf = getattr(d, attr)
    model, m, scored = train_catboost_region(cleandf, target='IBD')  # m = métricas de esa región
    all_metrics[region] = m

    # Mantener índice (SamplingOperations_code) y solo la predicción
    solo_ibd = scored[['IBD_pred']].copy()
    solo_ibd['region'] = region
    predicciones.append(solo_ibd)
    print(region)

predicciones = pd.concat(predicciones, axis=0)  # índice preservado
predicciones.index.name = 'SamplingOperations_code'

metrics_df = (pd.DataFrame.from_dict(all_metrics, orient='index')
                .reset_index()
                .rename(columns={'index':'region'}))
metrics_df
"""


18
5
4
10
22
9
21
20
12
8
3
17
14
11
13
19
1
7
6
16
15
2


,region,R2_train,R2_valid,MAE_train,MAE_valid,RMSE_train,RMSE_valid,best_iterations
0,18,0.999997,0.878967,0.002853,0.538452,1.149030e-05,0.665059,2770
1,5,0.999879,0.927646,0.022638,0.448746,7.502270e-04,0.519312,2996
2,4,1.000000,0.816799,0.000103,0.816917,1.472407e-08,1.409947,2641
3,10,0.999443,0.943318,0.055295,0.415377,4.575315e-03,0.455956,2999
4,22,1.000000,0.520372,0.000018,1.391299,4.069719e-10,4.031936,1255
5,9,0.992972,0.939320,0.119549,0.270092,2.315491e-02,0.196139,2996
6,21,0.999809,0.908663,0.030043,0.523856,1.320298e-03,0.675098,2999
7,20,0.999999,0.866621,0.001831,0.713329,4.406319e-06,0.845437,1638
8,12,0.998829,0.947046,0.064830,0.364278,6.326479e-03,0.299441,2999
9,8,1.000000,0.769914,0.000150,0.673607,3.010695e-08,1.002304,2906


In [ ]:
metrics_df.to_csv('metricts_dirty.csv',index=False)

In [ ]:
predicciones.to_csv('predicciones_dirty.csv',index = True)

In [ ]:
import numpy as np
import pandas as pd
from typing import Tuple, Dict, Any
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor

def train_catboost_region_tunueado(
    cleandf: pd.DataFrame,
    target: str = "IBD",
    test_size: float = 0.20,
    random_state: int = 42,
    cat_params: dict | None = None,
    use_ohe: bool = False,   # False = categóricas nativas (recomendado)
) -> Tuple[CatBoostRegressor, Dict[str, Any], pd.DataFrame]:
    """
    Entrena CatBoostRegressor para una región usando cleandf (con numéricas y categóricas).
    - Separa train (con target) y score (sin target)
    - Preprocesa (imputación cat). Si use_ohe=True, tú haces OHE fuera de esta función.
    - Entrena con early stopping y regularización
    - Devuelve: modelo, métricas en holdout, y scored_df (filas sin target con predicción)
    """

    # -----------------------------
    # 1) separar train / score
    # -----------------------------
    df_train = cleandf[cleandf[target].notna()].copy()
    df_score = cleandf[cleandf[target].isna()].copy()

    # columnas a tirar si existen (evitar leakage)
    drop_cols = [c for c in ["IBD", "IBD_EQR", "IBD_EQR_Status"] if c in cleandf.columns]

    X = df_train.drop(columns=drop_cols + [target], errors="ignore")
    y = df_train[target].astype(float)

    # split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # -----------------------------
    # 2) detectar numéricas / categóricas
    # -----------------------------
    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X_tr.select_dtypes(exclude=[np.number]).columns.tolist()

    # -----------------------------
    # 3) imputación ligera
    # -----------------------------
    # num: mediana no lo ghago por la estructura de taxones ; cat: literal '(missing)' (si usas OHE hazlo igual antes del OHE)
    X_tr_num = X_tr[num_cols].copy()
    X_te_num = X_te[num_cols].copy()


    X_tr_cat = X_tr[cat_cols].copy()
    X_te_cat = X_te[cat_cols].copy()
    X_tr_cat = X_tr_cat.fillna("(missing)")
    X_te_cat = X_te_cat.fillna("(missing)")

    if use_ohe:
        # -------------------------
        # 3.a) OHE (opcional)
        # -------------------------
        # Nota: si quieres OHE aquí, puedes hacerlo con pd.get_dummies para rapidez.
        # (Si ya traes OHE hecho fuera, simplemente deja use_ohe=False)
        X_tr_cat = pd.get_dummies(X_tr_cat, drop_first=False)
        X_te_cat = pd.get_dummies(X_te_cat, drop_first=False)
        # alinear columnas
        X_tr_cat, X_te_cat = X_tr_cat.align(X_te_cat, join="left", axis=1, fill_value=0)
        X_tr_proc = pd.concat([X_tr_num.reset_index(drop=True),
                               X_tr_cat.reset_index(drop=True)], axis=1)
        X_te_proc = pd.concat([X_te_num.reset_index(drop=True),
                               X_te_cat.reset_index(drop=True)], axis=1)
        cat_features = None  # ya no se usan índices de categóricas con OHE
    else:
        # -------------------------
        # 3.b) Categóricas nativas
        # -------------------------
        X_tr_proc = pd.concat([X_tr_num.reset_index(drop=True),
                               X_tr_cat.reset_index(drop=True)], axis=1)
        X_te_proc = pd.concat([X_te_num.reset_index(drop=True),
                               X_te_cat.reset_index(drop=True)], axis=1)
        # índices de columnas categóricas en el DataFrame concatenado
        # (van después de las numéricas)
        cat_features = list(range(len(num_cols), len(num_cols) + len(cat_cols)))

    # -----------------------------
    # 4) CatBoost: base + overrides
    # -----------------------------
    base_params = dict(
        loss_function="RMSE",
        eval_metric="RMSE",        # o "R2" si prefieres monitorear R2
        depth=5,                   # 4–6 para 2k x 200
        learning_rate=0.03,        # 0.02–0.05
        iterations=5000,           # early stopping cortará antes
        l2_leaf_reg=10,            # 8–12 reduce gap
        bagging_temperature=1.5,   # diversidad
        subsample=0.8,
        colsample_bylevel=0.7,
        random_strength=1.5,
        max_bin=128,
        random_state=random_state,
        verbose=0,
    )
    if cat_params:
        base_params.update(cat_params)

    model = CatBoostRegressor(**base_params)

    # -----------------------------
    # 5) entrenar con early stopping
    # -----------------------------
    fit_kwargs = dict(
        X=X_tr_proc,
        y=y_tr,
        eval_set=(X_te_proc, y_te),
        use_best_model=True,
    )
    if not use_ohe:
        fit_kwargs["cat_features"] = cat_features  # solo si usamos categóricas nativas

    model.fit(**fit_kwargs)

    # -----------------------------
    # 6) métricas
    # -----------------------------
    pred_tr = model.predict(X_tr_proc)
    pred_te = model.predict(X_te_proc)

    r2_tr = r2_score(y_tr, pred_tr)
    r2_te = r2_score(y_te, pred_te)
    rmse_tr = mean_squared_error(y_tr, pred_tr)
    rmse_te = mean_squared_error(y_te, pred_te)
    mae_tr = mean_absolute_error(y_tr, pred_tr)
    mae_te = mean_absolute_error(y_te, pred_te)

    metrics = {
        "R2_train": float(r2_tr),
        "R2_valid": float(r2_te),
        "MAE_train": float(mae_tr),
        "MAE_valid": float(mae_te),
        "RMSE_train": float(rmse_tr),
        "RMSE_valid": float(rmse_te),
        "best_iterations": int(model.get_best_iteration() or model.tree_count_),
        "gap": float(abs(r2_tr - r2_te)),
        "used_ohe": use_ohe,
    }

    # -----------------------------
    # 7) score para filas sin target
    # -----------------------------
    if not df_score.empty:
        Xs = df_score.drop(columns=drop_cols + [target], errors="ignore")

        Xs_num = Xs[num_cols].copy()
        Xs_cat = Xs[cat_cols].copy().fillna("(missing)")

        if use_ohe:
            Xs_cat = pd.get_dummies(Xs_cat, drop_first=False)
            # alinear con entrenamiento
            Xs_cat = Xs_cat.reindex(columns=X_tr_cat.columns, fill_value=0)

            Xs_proc = pd.concat([Xs_num.reset_index(drop=True),
                                 Xs_cat.reset_index(drop=True)], axis=1)
        else:
            Xs_proc = pd.concat([Xs_num.reset_index(drop=True),
                                 Xs_cat.reset_index(drop=True)], axis=1)

        df_score[target + "_pred"] = model.predict(Xs_proc)
        scored_df = df_score
    else:
        scored_df = pd.DataFrame(columns=list(cleandf.columns) + [target + "_pred"])

    return model, metrics, scored_df


In [ ]:
predicciones = []
all_metrics = {}  # guardará {region: métricas}

for region in regiones:
    attr = f"df_{int(region)}"           # df_1, df_2, df_18, ...
    if not hasattr(d, attr):
        # fallback por si tus nombres vienen con ceros: df_01
        attr = f"df_{str(region)}"
    if not hasattr(d, attr):
        raise AttributeError(f"No encontré {attr} en d")

    cleandf = getattr(d, attr)
    model, m, scored = train_catboost_region_tunueado(cleandf, target='IBD')  # m = métricas de esa región
    all_metrics[region] = m

    # Mantener índice (SamplingOperations_code) y solo la predicción
    solo_ibd = scored[['IBD_pred']].copy()
    solo_ibd['region'] = region
    predicciones.append(solo_ibd)
    print(region)

predicciones = pd.concat(predicciones, axis=0)  # índice preservado
predicciones.index.name = 'SamplingOperations_code'

metrics_df = (pd.DataFrame.from_dict(all_metrics, orient='index')
                .reset_index()
                .rename(columns={'index':'region'}))
metrics_df


18
5


KeyboardInterrupt: 

In [ ]:

def train_pred(
    cleandf: pd.DataFrame,
    target: str = 'IBD',
    random_state: int = 42,
    early_stopping_rounds: int = 200,
    cat_params: dict | None = None,
):
    """
    Entrena CatBoost para una región usando cleandf.
    - Separa train (con target) y score (sin target)
    - Preprocesa (imputación num/cat + OHE)
    - Entrena con early stopping
    - Regresa: modelo (Pipeline), métricas, scored_df (filas sin target con predicción)
    """
    # 1) separar train / score
    df_train = cleandf[cleandf[target].notna()].copy()
    df_score = cleandf[cleandf[target].isna()].copy()

    # X / y
    drop_cols = [c for c in ['IBD','IBD_EQR','IBD_EQR_Status'] if c in cleandf.columns]
    X = df_train.drop(columns=drop_cols, errors='ignore')
    y = df_train[target].astype(float)
   

    # 2) columnas num/cat
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    # preprocesamiento
    pre = ColumnTransformer([
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
        ]), num_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='(missing)')),
            ('ohe', OneHotEncoder(handle_unknown='ignore')),
        ]), cat_cols)
    ])

    # 3) modelo CatBoost (parámetros por defecto + override opcional)
    base_params = dict(
        depth=6, learning_rate=0.05, n_estimators=3000,
        loss_function='RMSE', random_state=random_state, verbose=0
    )
    if cat_params:
        base_params.update(cat_params)

    cb = Pipeline([
        ('pre', pre),
        ('model', CatBoostRegressor(**base_params))
    ])

    # 4) ajustar: primero ajustamos el preprocesador para armar matrices, luego el modelo con early stopping
    Xtr_proc = pre.fit_transform(X_tr)
    Xte_proc = pre.transform(X_te)
    cb.named_steps['model'].fit(
        Xtr_proc, y_tr,
        eval_set=(Xte_proc, y_te),
        use_best_model=True,
        early_stopping_rounds=early_stopping_rounds
    )

    # 5) métricas en hold-out y train (usando el pipeline para que transforme igual)
    r2_tr  = cb.score(X_tr, y_tr)
    r2_te  = cb.score(X_te, y_te)
    pred_tr = cb.predict(X_tr); pred_te = cb.predict(X_te)
    metrics = {
        'R2_train': r2_tr,
        'R2_valid': r2_te,
        'MAE_train': mean_absolute_error(y_tr, pred_tr),
        'MAE_valid': mean_absolute_error(y_te, pred_te),
        'RMSE_train': mean_squared_error(y_tr, pred_tr),
        'RMSE_valid': mean_squared_error(y_te, pred_te),
        'best_iterations': int(cb.named_steps['model'].get_best_iteration() or base_params['n_estimators'])
    }

    # 6) predicciones para filas sin target de esta región (si existen)
    if not df_score.empty:
        X_score = df_score.drop(columns=drop_cols, errors='ignore')
        df_score[target + '_pred'] = cb.predict(X_score)
        scored_df = df_score
        
    else:
        scored_df = pd.DataFrame(columns=list(cleandf.columns) + [target + '_pred'])

    return cb, metrics, scored_df